<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/misc/WIP-001-hft-sentiment-analysis-bert-imdb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HF Transformers - Sentiment Analysis - Bert/IMDB

First let's install the packages we'll be using:

In [1]:
!pip install transformers datasets evaluate accelerate

Let's retrieve the device we have available:

In [2]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


Let's load the [IMDB dataset](https://huggingface.co/datasets/stanfordnlp/imdb), which pairs IMDB reviews with a binary sentiment classification (positive, negative).

In [3]:
from datasets import load_dataset

raw_dataset = load_dataset("stanfordnlp/imdb")
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

The dataset has a `train` split with 25k samples, and a `test` split with 25k samples as well. There is also an extra `unsupervised` split with 50k entries, it's not obvious what that split is for, let's inspect it:

In [4]:
raw_dataset["unsupervised"][0]

{'text': 'This is just a precious little diamond. The play, the script are excellent. I cant compare this movie with anything else, maybe except the movie "Leon" wonderfully played by Jean Reno and Natalie Portman. But... What can I say about this one? This is the best movie Anne Parillaud has ever played in (See please "Frankie Starlight", she\'s speaking English there) to see what I mean. The story of young punk girl Nikita, taken into the depraved world of the secret government forces has been exceptionally over used by Americans. Never mind the "Point of no return" and especially the "La femme Nikita" TV series. They cannot compare the original believe me! Trash these videos. Buy this one, do not rent it, BUY it. BTW beware of the subtitles of the LA company which "translate" the US release. What a disgrace! If you cant understand French, get a dubbed version. But you\'ll regret later :)',
 'label': -1}

In [5]:
list(set([x["label"] for x in raw_dataset["unsupervised"]]))

[-1]

We've confirmed that `unsupervised` is what the name says, an unsupervised split that doesn't have any labels. It won't be useful for the sentiment analysis task, but could be useful for generating similar reviews for example (text generation task).

Let's learn more about the dataset features:

In [6]:
raw_dataset["train"].features

{'text': Value(dtype='string', id=None),
 'label': ClassLabel(names=['neg', 'pos'], id=None)}

Let's inspect some samples from the `train` split:

In [7]:
raw_dataset["train"][:5]["text"]

['I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, e

Let's check the variability of the text lengths:

In [8]:
raw_dataset = raw_dataset.map(lambda x: {"text_length" : len(x["text"].split())})
raw_dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'text_length'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'text_length'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'text_length'],
        num_rows: 50000
    })
})

In [9]:
min(raw_dataset["train"]["text_length"]), max(raw_dataset["train"]["text_length"]), sum(raw_dataset["train"]["text_length"]) / len(raw_dataset["train"]["text_length"])

(10, 2470, 233.7872)

Let's load a model to fine-tune:

In [10]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding

model_name = "openai-community/gpt2"#"bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

#tokenizer.pad_token = tokenizer.eos_token
if not tokenizer.pad_token_id: tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForSequenceClassification.from_pretrained(model_name)

model.config.pad_token_id = tokenizer.pad_token_id  # Ensure padding token is set

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at openai-community/gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


One thing, I'm not sure about is how the uncased model deals with casing. Can it not tokenized cased sequences? Or does it lowercase the input first? Let's check:

In [11]:
tokenizer.convert_ids_to_tokens(tokenizer("This is a Test!")["input_ids"])

['This', 'Ġis', 'Ġa', 'ĠTest', '!']

The model lowercases before tokenizing, which means we can feed it the dataset without lowercasing the text first. Let's now tokenize the dataset:

In [12]:
import multiprocessing

num_cores = multiprocessing.cpu_count()
print(f"Number of CPU cores: {num_cores}")

def _tokenize(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_dataset = raw_dataset.map(
    _tokenize,
    batched=True, # Enables batch processing
    num_proc=num_cores, # Adjust this based on your CPU cores
    remove_columns=["text", "text_length"] # Optional: reduces memory usage
)
tokenized_dataset

Number of CPU cores: 12


DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 50000
    })
})

Let's create a function to evaluate the model. This function will run through the test set and calculate the average prediction accuracy on it:

The starting accuracy is ~50%. This means that by default this model is pretty much random at being able to do sentiment classification. Let's setup a training run to fine-tune the model on this dataset:

In [13]:
import torch
import numpy as np
import evaluate
from transformers import Trainer
from transformers import TrainingArguments

# TODO: try to grok (use AdamW, adapt learning rate)
def compute_metrics(eval_pred):
    metric = evaluate.load("accuracy")
    logits, labels = eval_pred  # eval_pred is a tuple (logits, labels)
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="test",#"imdb_bert_fine_tuned",  # Output directory
    resume_from_checkpoint=True,  # This is the key line!
    num_train_epochs=3,  # Adjust as needed
    per_device_train_batch_size=32, # Start with a large batch size
    per_device_eval_batch_size=32, # Start with a large batch size
    gradient_accumulation_steps=1,  # Adjust if you run out of memory
    fp16=False, # Try bf16 first!
    bf16=True, # Brain Floating Point for A100 - try this first
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    #learning_rate=1e-7,  # Adjust as needed

    learning_rate=2e-5,  # Start with this, then tune
    warmup_ratio=0.1,  # Adjust as needed
    weight_decay=0.01,  # Adjust as needed
    adam_epsilon=1e-8,
    max_grad_norm=1.0,
    #warmup_steps=0,
    logging_steps=50, # Log every 50 steps,
    eval_steps=5 * len(tokenized_dataset["train"]) // 64#training_args.per_device_train_batch_size # Evaluate every 5 epochs
)

trainer = Trainer(
    model, # the instantiated 🤗 Transformers model to be trained
    training_args, # training arguments, defined above
    train_dataset=tokenized_dataset["train"], # The dataset to train the model on
    eval_dataset=tokenized_dataset["test"], # The dataset to evaluate the model on
    data_collator=data_collator, # defaults to DataCollatorWithPadding if not provided
    tokenizer=tokenizer, # The tokenizer to be used
    compute_metrics=compute_metrics  # Add the compute_metrics function
)

<ipython-input-13-f459217e91e4>:39: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [14]:
import evaluate

def eval():
  true_labels = tokenized_dataset["test"]['label']  # Ensure this key matches your dataset
  predictions = trainer.predict(tokenized_dataset["test"])
  logits = predictions.predictions
  predicted_labels = np.argmax(logits, axis=1)  # Convert logits to predicted labels
  metric = evaluate.load("accuracy")
  accuracy = metric.compute(predictions=predicted_labels, references=true_labels)
  return accuracy

In [15]:
eval()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: tsilva to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


{'accuracy': 0.5}

In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss,Model Preparation Time,Accuracy
1,0.198800,0.165920,0.003100,0.936200
2,0.151000,0.159655,0.003100,0.943040


KeyboardInterrupt: 

In [ ]:
eval()

Now that the model is trained, let's run the evaluation again:

In [ ]:
tokenized = tokenizer([
    "I have *had* it with these motherfucking *snakes* on this motherfucking *plane*!",
    "Spam, Spam, Spam, Spam! Spam, Spam, Spam, Spam! Lovely Spam, wonderful Spam!"
], padding=True, truncation=True, return_tensors="pt").to(DEVICE)
output = model(**tokenized)
output.logits.shape, output.logits

In [ ]:
predictions = torch.argmax(output.logits, dim=1).cpu().numpy()
predictions

In [ ]:
LABELS = raw_dataset["train"].features["label"].names
[LABELS[x] for x in predictions]

In [ ]:
misclassified_indices = np.where(predicted_labels != true_labels)[0]
misclassified_indices

In [ ]:
misclassified_sample = tokenized_dataset["test"][int(misclassified_indices[0])]
misclassified_true_label = misclassified_sample["label"]
misclassified_true_label

In [ ]:
from transformers import AutoConfig

model_name = "bert-base-uncased"  # Replace with your model
config = AutoConfig.from_pretrained(model_name)
print(config.max_position_embeddings)  # Max tokens the model can handle

In [ ]:
decoded_text = tokenizer.decode(misclassified_sample["input_ids"], skip_special_tokens=True)
print(decoded_text)

In [ ]:
result = model(input_ids=torch.tensor([misclassified_sample["input_ids"]]).to(DEVICE), attention_mask=torch.tensor([misclassified_sample["attention_mask"]]).to(DEVICE))
result

In [ ]:
result = torch.argmax(result.logits, dim=1).cpu().numpy()[0]
LABELS[result]

NOTE: chatgpt can classify this sentence correctly
TODO: extract validation split and run eval on test split